In [1]:
! git clone https://github.com/Ajax0564/Tritonml.git

Cloning into 'Tritonml'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 181 (delta 75), reused 160 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 263.74 KiB | 3.42 MiB/s, done.
Resolving deltas: 100% (75/75), done.


In [2]:
import sys
sys.path.append("../working/Tritonml/src")

In [3]:
from ops.linear_rms import TritonLinearRMSNormLayer
from ops.mlp_gelu import TritonGeluMlpLayer
from ops.linear import TritonLinearLayer
from ops.rope import TritonRopeLayer
from ops.rms import TritonRMSNormLayer
from ops.encoder_varlen_attention import TritonVerlenAttention

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import math
import torch

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, base=10000):
        super().__init__()
        self.head_dim = head_dim

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, position_ids):
        """
        position_ids: [T]
        Returns:
            cos, sin: [T, head_dim]
        """
        freqs = torch.outer(position_ids.float(), self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        return emb.cos(), emb.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def build_position_ids(cu_seqlens):
    """
    Builds per-sequence reset positional indices.

    Example:
        lengths = [5,3,7]
        returns:
        tensor([0,1,2,3,4, 0,1,2, 0,1,2,3,4,5,6])
    """
    lengths = cu_seqlens[1:] - cu_seqlens[:-1]
    return torch.cat([
        torch.arange(l, device=cu_seqlens.device)
        for l in lengths
    ])

class RoPE(nn.Module):
    def __init__(self, head_dim=64, max_seq_len=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq) 
        
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos(), persistent=False) # [Max_Seq, Dh]
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, x, seq_idx):
        # x: [T, H, Dh]
        cos = self.cos[seq_idx].unsqueeze(1) # [T, 1, Dh]
        sin = self.sin[seq_idx].unsqueeze(1)
        
        x1 = x[..., :x.shape[-1]//2]
        x2 = x[..., x.shape[-1]//2:]
        x_rotated = torch.cat((-x2, x1), dim=-1)
        
        return (x * cos) + (x_rotated * sin)
        

class VarLenMHA(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        assert dim % num_heads == 0

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads

        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.out_proj = TritonLinearRMSNormLayer(dim, dim)

        self.rope = RoPE(self.head_dim)
        self.attn = TritonVerlenAttention(self.dim**-0.5)
    def forward(self, x,position_ids, cu_seqlens):
        """
        x: [T, D]
        cu_seqlens: [B+1]
        """
        T = x.size(0)
        device = x.device

       
        qkv = self.qkv(x)                     # [T, 3D]
        qkv = qkv.view(T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=1)           # each: [T, H, Dh]

        # position_ids = build_position_ids(cu_seqlens)  # [T]
        assert position_ids.numel() == T, f"Position IDs length {position_ids.numel()} != T {T}"
        q = self.rope(q, position_ids)
        k = self.rope(k, position_ids)
        cu_seqlens = cu_seqlens.to(torch.int32).contiguous()
    

        out =  self.attn(q.contiguous(),k.contiguous(),v.contiguous(),cu_seqlens)

       
        out = out.view(T, self.dim)

        return self.out_proj(out)

class VarLenEncoderLayer(nn.Module):
    def __init__(self, dim, num_heads, hidden_dim):
        super().__init__()

        self.attn = VarLenMHA(dim, num_heads)

        # self.ffn = TritonGeluMlpLayer(dim, hidden_dim,dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, dim),
        )

        self.norm1 = TritonRMSNormLayer(dim)
        self.norm2 = TritonRMSNormLayer(dim)

    def forward(self, x,position_ids, cu_seqlens):
        x = x + self.attn(self.norm1(x),position_ids, cu_seqlens)
        x = x + self.ffn(self.norm2(x))
        return x


class VarLenTransformerEncoder(nn.Module):
    def __init__(self, dim=768, num_heads=12, hidden_dim=3072, num_layers=6):
        super().__init__()
        self.word_embeddings = nn.Embedding(50265,768)
        

        self.layers = nn.ModuleList([
            VarLenEncoderLayer(dim, num_heads, hidden_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x, cu_seqlens):
        position_ids = build_position_ids(cu_seqlens)
        x = self.word_embeddings(x)
        for layer in self.layers:
            x = layer(x,position_ids, cu_seqlens)
        return x

class FlashVarLenClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.encoder = VarLenTransformerEncoder()

        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x, cu_seqlens):
        x = self.encoder( x, cu_seqlens)
        first_token_indices = cu_seqlens[:-1]
        cls_tokens = x[first_token_indices]   # [B, D]

        logits = self.classifier(cls_tokens)  # [B, num_classes]
        return logits

In [5]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd

In [6]:
data_path = '../input/data-for-distilation' 
train = pd.read_csv('/kaggle/input/notebooks/ajax0564/data-for-distilation/Clinc_Train.csv')
val = pd.read_csv('/kaggle/input/notebooks/ajax0564/data-for-distilation/Clinc_valid.csv')
n_classes = np.unique(train.Target).shape[0]
train.head(2)


,Text,Target,intent
0,what expression would i use to say i love you ...,61,translate
1,can you tell me how to say 'i do not speak muc...,61,translate


In [7]:
n_classes

151

In [8]:
from torch.utils.data import Dataset
import torch
model_ckpt = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt) 

class ClinicDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=128):
        self.texts = data['Text'].values
        self.targets = data['Target'].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_attention_mask=False
        )

        input_ids = torch.tensor(encoding['input_ids'], dtype=torch.long)

        return {
            'input_ids': input_ids,
            'seqlen': input_ids.size(0),
            'target': torch.tensor(self.targets[idx], dtype=torch.long)
        }

    def __len__(self):
        return len(self.targets)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def collate_varlen(batch):
    input_ids_list = [item['input_ids'] for item in batch]
    seqlens = torch.tensor([item['seqlen'] for item in batch], dtype=torch.long)
    targets = torch.tensor([item['target'] for item in batch], dtype=torch.long)

    # Concatenate tokens into 1D packed tensor
    input_ids = torch.cat(input_ids_list, dim=0)

    # Compute cumulative sequence lengths (important!)
    cu_seqlens = torch.zeros(len(seqlens) + 1, dtype=torch.long)
    cu_seqlens[1:] = torch.cumsum(seqlens, dim=0)

    return {
        'input_ids': input_ids,       # [total_tokens]           # [B]
        'cu_seqlens': cu_seqlens,     # [B+1]
        'targets': targets            # [B]
    }


In [10]:
train_loader = torch.utils.data.DataLoader(ClinicDataset(train, tokenizer),batch_size=32, shuffle=True, num_workers=2,collate_fn=collate_varlen)
val_loader = torch.utils.data.DataLoader(ClinicDataset(val, tokenizer),batch_size=32, shuffle=True, num_workers=2,collate_fn=collate_varlen)

In [11]:
tokenizer

RobertaTokenizerFast(name_or_path='roberta-base', vocab_size=50265, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
}
)

In [12]:
model = FlashVarLenClassifier(n_classes)
model.to('cuda')

FlashVarLenClassifier(
  (encoder): VarLenTransformerEncoder(
    (word_embeddings): Embedding(50265, 768)
    (layers): ModuleList(
      (0-5): 6 x VarLenEncoderLayer(
        (attn): VarLenMHA(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (out_proj): TritonLinearRMSNormLayer()
          (rope): RoPE()
          (attn): TritonVerlenAttention()
        )
        (ffn): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
        (norm1): TritonRMSNormLayer(hidden_size=768, eps=1e-06)
        (norm2): TritonRMSNormLayer(hidden_size=768, eps=1e-06)
      )
    )
  )
  (classifier): Linear(in_features=768, out_features=151, bias=True)
)

In [13]:
def valid_func(model,val_loader,val_bar):
    model.eval()
    loss_fn = torch.nn.CrossEntropyLoss()
    PROB = []
    TARGETS = []
    losses = []
    PREDS = []
   
    for batch_idx,data in enumerate(val_loader):
        val_bar.update(1)
        input_ids = data['input_ids'].cuda()
        cu_seqlens = data['cu_seqlens'].cuda()
        targets = data['targets'].long().view(-1).cuda()
        pred = model(input_ids,cu_seqlens)
        with torch.no_grad():
             logits = model(input_ids,cu_seqlens)

        PREDS += [torch.argmax(logits, 1).detach().cpu()]
        TARGETS += [targets.detach().cpu()]

        loss = loss_fn (logits, targets)
        losses.append(loss.item())
        val_bar.set_description(f'step: {batch_idx+1} loss: {"%.4f" % loss}')

    PREDS = torch.cat(PREDS).cpu().numpy()
    TARGETS = torch.cat(TARGETS).cpu().numpy()
    accuracy = (PREDS==TARGETS).mean()
   
    loss_valid = np.mean(losses)
    return loss_valid, accuracy

In [14]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-5)  
EPOCHS  = 2
num_train_optimization_steps = int(EPOCHS * len(train_loader))

epoch_check = len(train_loader)
total_step = epoch_check*EPOCHS
train_bar = tqdm(total=total_step, dynamic_ncols=True)
val_bar = tqdm(total=len(val_loader),leave = True, dynamic_ncols=True)
t_step = 1
for epoch in range(EPOCHS):
    avg_loss = 0.0
    model.train()
    loss_list = []
    for step, data in enumerate(train_loader):
        train_bar.update(1)
        input_ids = data['input_ids'].cuda()
        cu_seqlens = data['cu_seqlens'].cuda()
        targets = data['targets'].long().view(-1).cuda()
        pred = model(input_ids,cu_seqlens)
        loss = loss_fn(pred, targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_bar.set_description(f'epoch: {epoch+1} step: {t_step} loss: {"%.4f" % loss}')
        t_step+=1
        loss_list.append(loss.detach().cpu().item())
    avg_loss = np.round(np.mean(loss_list), 4)
    vloss,vaccuracy = valid_func(model,val_loader,val_bar)
    print(f'Epoc: {epoch} loss: {"%.4f" % vloss},accuracy: {"%.4f" % vaccuracy}')
    val_bar.reset()
   

  0%|          | 0/954 [00:00<?, ?it/s]

  0%|          | 0/97 [00:00<?, ?it/s]

/tmp/ipykernel_57/147053086.py:25: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  train_bar.set_description(f'epoch: {epoch+1} step: {t_step} loss: {"%.4f" % loss}')


Epoc: 0 loss: 1.1622,accuracy: 0.7342
Epoc: 1 loss: 0.8315,accuracy: 0.7958
